In [0]:
import boto3
import json
from pyspark.sql.functions import current_timestamp, lit, input_file_name

# 1. Cargar Credenciales
try:
    with open("aws_secrets.json", "r") as f:
        config = json.load(f)
    print("✅ Credenciales listas.")
except:
    print("❌ Error cargando secretos.")

# 2. Configurar Cliente Boto3
s3_client = boto3.client(
    's3',
    aws_access_key_id=config['aws_access_key'],
    aws_secret_access_key=config['aws_secret_key'], 
    region_name='us-east-1'
)

bucket_name = "bronce-scrap-date"

# ==========================================
# FUNCIÓN 1: DESCUBRIMIENTO DE FUENTES
# ==========================================
def obtener_fuentes_disponibles(bucket):
    print(f"📡 Escaneando carpetas en 'raw/'...")
    # El truco es usar Delimiter='/', AWS lo interpreta como carpetas
    response = s3_client.list_objects_v2(Bucket=bucket, Prefix='raw/', Delimiter='/')
    
    fuentes_encontradas = []
    
    if 'CommonPrefixes' in response:
        for prefix in response['CommonPrefixes']:
            # El prefix viene como 'raw/fincaraiz/', hay que limpiarlo
            nombre_sucio = prefix['Prefix'] 
            nombre_limpio = nombre_sucio.replace("raw/", "").replace("/", "")
            fuentes_encontradas.append(nombre_limpio)
            
    print(f"✅ Se encontraron {len(fuentes_encontradas)} fuentes: {fuentes_encontradas}")
    return fuentes_encontradas

# ==========================================
# FUNCIÓN 2: PROCESAMIENTO (Boto3 + Spark)
# ==========================================
def procesar_dinamico(fuente, bucket):
    # Buscamos en la subcarpeta batches
    prefix = f"raw/{fuente}/batches/"
    
    # Listamos archivos reales
    response = s3_client.list_objects_v2(Bucket=bucket, Prefix=prefix)
    
    if 'Contents' not in response:
        print(f"⏩ {fuente}: Carpeta existe pero está vacía (Sin batches).")
        return

    # Filtramos solo Parquet
    archivos_parquet = [obj['Key'] for obj in response['Contents'] if obj['Key'].endswith(".parquet")]
    
    if not archivos_parquet:
        print(f"⏩ {fuente}: No tiene archivos .parquet todavía.")
        return

    print(f"\n🚀 {fuente.upper()}: Procesando {len(archivos_parquet)} archivos...")
    
    for archivo_key in archivos_parquet:
        ruta_s3a = f"s3a://{bucket}/{archivo_key}"
        ruta_bronze = f"s3a://{bucket}/bronze/{fuente}/"
        
        try:
            # Spark entra con ruta exacta
            df_raw = spark.read.format("parquet") \
                .option("fs.s3a.access.key", config['aws_access_key']) \
                .option("fs.s3a.secret.key", config['aws_secret_key']) \
                .option("fs.s3a.endpoint", "s3.amazonaws.com") \
                .load(ruta_s3a)
            
            # Guardado
            df_raw \
                .withColumn("bronze_ingested_at", current_timestamp()) \
                .withColumn("source_system", lit(fuente)) \
                .write.format("delta") \
                .mode("append") \
                .option("mergeSchema", "true") \
                .option("fs.s3a.access.key", config['aws_access_key']) \
                .option("fs.s3a.secret.key", config['aws_secret_key']) \
                .save(ruta_bronze)
                
            print(f"   Success: {archivo_key.split('/')[-1]}")
            
        except Exception as e:
            print(f"   Error: {str(e)}")

# ==========================================
# 3. EJECUCIÓN AUTOMÁTICA
# ==========================================
fuentes_detectadas = obtener_fuentes_disponibles(bucket_name)

for fuente in fuentes_detectadas:
    procesar_dinamico(fuente, bucket_name)

print("\n🏁 Barrido completo de todas las carpetas.")